<a href="https://colab.research.google.com/github/WiMean/Project/blob/main/NewsMiner_train_v4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NewsMiner v4: обучение классификатора (11 классов, заголовок + описание)

**Что это.** Дообучение `DeepPavlov/rubert-base-cased` на наших данных:
5757 обучающих, 903 валидационных,
903 тестовых строк и **2194 псевдо-размеченных** строк
из неразмеченных пулов (уверенность текущей модели ≥ 0.85).

**Почему так:**
- описание входа заметно помогает на коротких заголовках, часть описаний
  случайно отбрасывается, чтобы модель работала и на источниках без описания;
- веса классов + label smoothing — редкие классы (Здоровье, Лайфстайл, Другие)
  иначе «съедаются» крупными; вес ограничен сверху, иначе служебный класс
  «Другие» получает ~6.4 и отбирает ёмкость у содержательных;
- три сида: берём лучшую модель по val macro-F1, это устойчивее одного прогона;
- перед выдачей архива проверяется, что int8 повторяет PyTorch, и отдельно
  измеряется точность самого int8 — её и увидит приложение.

**Время:** 25–50 минут на бесплатной T4.

**Что прислать назад:** скачанный `newsminer_model.zip` — я поставлю модель в
приложение, пересоберу APK и замерю качество и скорость на телефоне.

In [2]:
!pip -q install "transformers>=4.40,<5" onnx onnxruntime
import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.30.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.42.0 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.
torch 2.11.0+cu130 | cuda True | Tesla T4


In [3]:
import base64, collections, csv, gzip, io, json, os, random, time, zipfile
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
CATS = ["Политика и геополитика", "Экономика и бизнес", "Технологии и ИТ", "Наука и космос", "Здоровье и медицина", "Лайфстайл и быт", "Спорт и киберспорт", "Культура и шоу-бизнес", "Происшествия и криминал", "Общество и социальная сфера", "Другие"]
LABEL = {c: i for i, c in enumerate(CATS)}
MAXLEN, BATCH, ACCUM, EPOCHS, LR, DESC_DROP = 96, 32, 1, 5, 2e-5, 0.3
MODEL = 'DeepPavlov/rubert-base-cased'
SEEDS = [42, 7, 2026]
USE_PSEUDO = True

PACKED = "..."
rows = list(csv.DictReader(io.StringIO(gzip.decompress(base64.b64decode(PACKED)).decode('utf-8'))))
splits = {s: [r for r in rows if r['split'] == s and r['category'] in LABEL] for s in ('train', 'val', 'test', 'pseudo')}
train_rows = splits['train'] + (splits['pseudo'] if USE_PSEUDO else [])
print('данные:', {k: len(v) for k, v in splits.items()}, '| в обучении:', len(train_rows))
print('распределение обучения:', collections.Counter(r['category'] for r in train_rows).most_common())

данные: {'train': 5757, 'val': 903, 'test': 903, 'pseudo': 2194} | в обучении: 7951
распределение обучения: [('Политика и геополитика', 1256), ('Происшествия и криминал', 1228), ('Экономика и бизнес', 986), ('Культура и шоу-бизнес', 733), ('Технологии и ИТ', 730), ('Общество и социальная сфера', 689), ('Лайфстайл и быт', 630), ('Спорт и киберспорт', 615), ('Наука и космос', 531), ('Здоровье и медицина', 502), ('Другие', 51)]


In [4]:
# Модель публичная: ни аккаунт, ни токен Hugging Face не нужны.
# token=False явно говорит библиотеке не искать секреты и не просить вход.
tok = AutoTokenizer.from_pretrained(MODEL, token=False)

class DS(Dataset):
    def __init__(self, rows, train=False, rng=None):
        self.labels = [LABEL[r['category']] for r in rows]
        self.texts = []
        for r in rows:
            t, d = r['title'].strip(), (r['description'] or '').strip()
            drop = train and rng.random() < DESC_DROP
            self.texts.append(t if (drop or not d) else f'{t}. {d}')
    def __len__(self): return len(self.labels)
    def __getitem__(self, i): return self.texts[i], self.labels[i]

def collate(batch):
    texts, labels = zip(*batch)
    enc = tok(list(texts), truncation=True, max_length=MAXLEN, padding=True, return_tensors='pt')
    return {k: v.to(DEVICE) for k, v in enc.items()}, torch.tensor(labels).to(DEVICE)

def macro_f1(y_true, y_pred, n=len(CATS)):
    f1s, precs, recs = [], [], []
    for c in range(n):
        tp = sum(1 for t, p in zip(y_true, y_pred) if t == c and p == c)
        fp = sum(1 for t, p in zip(y_true, y_pred) if t != c and p == c)
        fn = sum(1 for t, p in zip(y_true, y_pred) if t == c and p != c)
        pr = tp / (tp + fp) if tp + fp else 0.0
        rc = tp / (tp + fn) if tp + fn else 0.0
        f1s.append(2 * pr * rc / (pr + rc) if pr + rc else 0.0)
    return sum(f1s) / n, f1s

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    preds, trues = [], []
    for enc, y in loader:
        logits = model(**enc).logits
        preds.extend(logits.argmax(-1).cpu().tolist())
        trues.extend(y.cpu().tolist())
    acc = sum(p == t for p, t in zip(preds, trues)) / len(trues)
    f1, _ = macro_f1(trues, preds)
    return acc, f1, preds, trues

tokenizer_config.json:   0%|          | 0.00/24.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/642 [00:00<?, ?B/s]

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [5]:
def train_one(seed, train_rows, val_rows, epochs=EPOCHS, verbose=True):
    torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)
    rng = np.random.default_rng(seed)
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL, num_labels=len(CATS), token=False).to(DEVICE)
    tr = DataLoader(DS(train_rows, train=True, rng=rng), batch_size=BATCH, shuffle=True, collate_fn=collate)
    va = DataLoader(DS(val_rows), batch_size=64, collate_fn=collate)

    cnt = collections.Counter(LABEL[r['category']] for r in train_rows)
    w = torch.tensor([len(train_rows) / (len(CATS) * max(1, cnt[i])) for i in range(len(CATS))], dtype=torch.float).to(DEVICE)
    w = w / w.mean()
    # Потолок веса. Без него «Другие» (служебный класс, 8 строк в тесте) получает
    # вес ~6.4 и отбирает ёмкость у содержательных классов, а редкие классы
    # начинают переучиваться на своих же ошибках.
    w = w.clamp(max=4.0)
    lossf = nn.CrossEntropyLoss(weight=w, label_smoothing=0.05)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    steps = max(1, len(tr) // ACCUM) * epochs
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, total_steps=steps, pct_start=0.1)

    best = {'f1': -1, 'state': None, 'epoch': 0, 'acc': 0}
    for ep in range(1, epochs + 1):
        model.train(); t0 = time.time(); run = 0.0; opt.zero_grad()
        for i, (enc, y) in enumerate(tr):
            out = model(**enc)
            loss = lossf(out.logits, y) / ACCUM
            loss.backward(); run += loss.item() * ACCUM
            if (i + 1) % ACCUM == 0 or i + 1 == len(tr):
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step(); sched.step(); opt.zero_grad()
            if verbose and i % 100 == 0:
                print(f'  сид {seed} эпоха {ep} шаг {i}/{len(tr)} loss {run/(i+1):.4f} ({time.time()-t0:.0f}s)', flush=True)
        acc, f1, _, _ = evaluate(model, va)
        print(f'сид {seed} эпоха {ep}: val acc {acc:.4f} | val macro-F1 {f1:.4f} | {time.time()-t0:.0f}s', flush=True)
        if f1 > best['f1']:
            best = {'f1': f1, 'acc': acc, 'epoch': ep,
                    'state': {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}}
    model.load_state_dict(best['state'])
    print(f'сид {seed}: лучшая эпоха {best["epoch"]}, val acc {best["acc"]:.4f}, val macro-F1 {best["f1"]:.4f}', flush=True)
    return model, best

results = {}
for seed in SEEDS:
    print(f'\n=== сид {seed} ===', flush=True)
    model, best = train_one(seed, train_rows, splits['val'])
    results[seed] = (model, best)
best_seed = max(results, key=lambda s: results[s][1]['f1'])
model, best = results[best_seed]
print(f'\nЛУЧШИЙ СИД: {best_seed} (val macro-F1 {best["f1"]:.4f})')


=== сид 42 ===


pytorch_model.bin:   0%|          | 0.00/714M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at DeepPavlov/rubert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  сид 42 эпоха 1 шаг 0/249 loss 2.5186 (4s)
  сид 42 эпоха 1 шаг 100/249 loss 2.2940 (51s)
  сид 42 эпоха 1 шаг 200/249 loss 1.7299 (104s)
сид 42 эпоха 1: val acc 0.8328 | val macro-F1 0.7635 | 136s
  сид 42 эпоха 2 шаг 0/249 loss 1.2975 (0s)
  сид 42 эпоха 2 шаг 100/249 loss 0.8065 (54s)
  сид 42 эпоха 2 шаг 200/249 loss 0.7751 (107s)
сид 42 эпоха 2: val acc 0.8494 | val macro-F1 0.7906 | 138s
  сид 42 эпоха 3 шаг 0/249 loss 0.5548 (1s)
  сид 42 эпоха 3 шаг 100/249 loss 0.6080 (54s)
  сид 42 эпоха 3 шаг 200/249 loss 0.6082 (108s)
сид 42 эпоха 3: val acc 0.8571 | val macro-F1 0.8077 | 139s
  сид 42 эпоха 4 шаг 0/249 loss 0.4217 (0s)
  сид 42 эпоха 4 шаг 100/249 loss 0.5143 (54s)
  сид 42 эпоха 4 шаг 200/249 loss 0.5180 (109s)
сид 42 эпоха 4: val acc 0.8749 | val macro-F1 0.8298 | 140s
  сид 42 эпоха 5 шаг 0/249 loss 0.4749 (1s)
  сид 42 эпоха 5 шаг 100/249 loss 0.5035 (55s)
  сид 42 эпоха 5 шаг 200/249 loss 0.4919 (108s)
сид 42 эпоха 5: val acc 0.8738 | val macro-F1 0.8212 | 138s
сид 4

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at DeepPavlov/rubert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  сид 7 эпоха 1 шаг 0/249 loss 2.5529 (1s)
  сид 7 эпоха 1 шаг 100/249 loss 2.2958 (55s)
  сид 7 эпоха 1 шаг 200/249 loss 1.7280 (110s)
сид 7 эпоха 1: val acc 0.8295 | val macro-F1 0.7797 | 140s
  сид 7 эпоха 2 шаг 0/249 loss 1.2507 (0s)
  сид 7 эпоха 2 шаг 100/249 loss 0.7813 (55s)
  сид 7 эпоха 2 шаг 200/249 loss 0.7820 (108s)
сид 7 эпоха 2: val acc 0.8638 | val macro-F1 0.8043 | 139s
  сид 7 эпоха 3 шаг 0/249 loss 0.4683 (1s)
  сид 7 эпоха 3 шаг 100/249 loss 0.6172 (55s)
  сид 7 эпоха 3 шаг 200/249 loss 0.6104 (109s)
сид 7 эпоха 3: val acc 0.8738 | val macro-F1 0.8190 | 139s
  сид 7 эпоха 4 шаг 0/249 loss 0.4982 (0s)
  сид 7 эпоха 4 шаг 100/249 loss 0.5417 (54s)
  сид 7 эпоха 4 шаг 200/249 loss 0.5337 (107s)
сид 7 эпоха 4: val acc 0.8704 | val macro-F1 0.8220 | 138s
  сид 7 эпоха 5 шаг 0/249 loss 0.4485 (0s)
  сид 7 эпоха 5 шаг 100/249 loss 0.5005 (54s)
  сид 7 эпоха 5 шаг 200/249 loss 0.4995 (108s)
сид 7 эпоха 5: val acc 0.8726 | val macro-F1 0.8298 | 138s
сид 7: лучшая эпоха 5, va

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at DeepPavlov/rubert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  сид 2026 эпоха 1 шаг 0/249 loss 2.5591 (1s)
  сид 2026 эпоха 1 шаг 100/249 loss 2.3019 (54s)
  сид 2026 эпоха 1 шаг 200/249 loss 1.7363 (108s)
сид 2026 эпоха 1: val acc 0.8217 | val macro-F1 0.7359 | 138s
  сид 2026 эпоха 2 шаг 0/249 loss 0.7588 (0s)
  сид 2026 эпоха 2 шаг 100/249 loss 0.8144 (53s)
  сид 2026 эпоха 2 шаг 200/249 loss 0.7841 (108s)
сид 2026 эпоха 2: val acc 0.8538 | val macro-F1 0.7923 | 138s
  сид 2026 эпоха 3 шаг 0/249 loss 0.6957 (0s)
  сид 2026 эпоха 3 шаг 100/249 loss 0.6382 (54s)
  сид 2026 эпоха 3 шаг 200/249 loss 0.6208 (109s)
сид 2026 эпоха 3: val acc 0.8627 | val macro-F1 0.8097 | 140s
  сид 2026 эпоха 4 шаг 0/249 loss 0.6381 (1s)
  сид 2026 эпоха 4 шаг 100/249 loss 0.5278 (54s)
  сид 2026 эпоха 4 шаг 200/249 loss 0.5322 (108s)
сид 2026 эпоха 4: val acc 0.8760 | val macro-F1 0.8237 | 139s
  сид 2026 эпоха 5 шаг 0/249 loss 0.6085 (0s)
  сид 2026 эпоха 5 шаг 100/249 loss 0.4868 (55s)
  сид 2026 эпоха 5 шаг 200/249 loss 0.4958 (109s)
сид 2026 эпоха 5: val acc 0

In [6]:
# Честный замер на отложенном тесте (его модель не видела)
te = DataLoader(DS(splits['test']), batch_size=64, collate_fn=collate)
acc, f1, preds, trues = evaluate(model, te)
per = macro_f1(trues, preds)[1]
print(f'ТЕСТ: acc {acc:.4f} | macro-F1 {f1:.4f} | строк {len(trues)}')
for c, f in zip(CATS, per):
    n = sum(1 for t in trues if t == LABEL[c])
    cor = sum(1 for t, p in zip(trues, preds) if t == LABEL[c] and p == LABEL[c])
    print(f'  {c:32s} n={n:3d} recall={cor/max(1,n):.3f} F1={f:.2f}')
conf = collections.Counter((CATS[t], CATS[p]) for t, p in zip(trues, preds) if t != p)
print('\nчастые ошибки:')
for (a, b), k in conf.most_common(12):
    print(f'  {k:3d}  {a[:28]:30s} → {b[:28]}')

ТЕСТ: acc 0.8472 | macro-F1 0.7717 | строк 903
  Политика и геополитика           n=198 recall=0.909 F1=0.91
  Экономика и бизнес               n=113 recall=0.832 F1=0.86
  Технологии и ИТ                  n= 89 recall=0.876 F1=0.89
  Наука и космос                   n= 53 recall=0.755 F1=0.82
  Здоровье и медицина              n= 37 recall=0.838 F1=0.78
  Лайфстайл и быт                  n= 71 recall=0.761 F1=0.79
  Спорт и киберспорт               n= 82 recall=0.951 F1=0.95
  Культура и шоу-бизнес            n= 50 recall=0.820 F1=0.78
  Происшествия и криминал          n=133 recall=0.872 F1=0.85
  Общество и социальная сфера      n= 69 recall=0.754 F1=0.78
  Другие                           n=  8 recall=0.125 F1=0.08

частые ошибки:
    8  Политика и геополитика         → Происшествия и криминал
    7  Общество и социальная сфера    → Происшествия и криминал
    6  Лайфстайл и быт                → Культура и шоу-бизнес
    6  Технологии и ИТ                → Экономика и бизнес
    6 

## (необязательно) добить качество на train+val

Если тест выше 85% — можно пропустить. Если чуть ниже, запустите эту ячейку:
она переобучает с нуля на `train + val + pseudo` и заново мерит тест.

Эпохи берутся из лучшего сида, а для выбора лучшей эпохи откладывается
стратифицированные 10% от объединённой выборки. Раньше валидация входила
в обучение и по ней же выбиралась эпоха — метрика росла до 1.0, выбиралась
последняя эпоха, и тест после «дообучения» становился хуже, чем до него.

In [7]:
USE_TRAINVAL = False   # поставьте True, чтобы включить

if USE_TRAINVAL:
    full = splits['train'] + splits['val'] + (splits['pseudo'] if USE_PSEUDO else [])
    # ВАЖНО, здесь была ошибка. Раньше стояло train_one(best_seed, full, splits['val']):
    # валидация входила в full, то есть в обучение, и по ней же выбиралась лучшая
    # эпоха. Метрика при этом росла почти до 1.0, выбиралась последняя эпоха,
    # и тест после такого «дообучения» становился хуже, чем до него.
    # Теперь откладываем стратифицированный кусок именно для выбора эпохи.
    rng = np.random.default_rng(best_seed)
    by_cat = collections.defaultdict(list)
    for r in full:
        by_cat[r['category']].append(r)
    sel, rest = [], []
    for cat, rs in by_cat.items():
        rs = list(rs); rng.shuffle(rs)
        k = max(1, int(round(len(rs) * 0.1)))
        sel += rs[:k]; rest += rs[k:]
    print(f'обучение {len(rest)}, выбор эпохи {len(sel)} (стратифицировано, 10%)')
    model, best = train_one(best_seed, rest, sel, epochs=best['epoch'])
    acc, f1, preds, trues = evaluate(model, te)
    per = macro_f1(trues, preds)[1]
    print(f'ПОСЛЕ ДООБУЧЕНИЯ НА train+val: acc {acc:.4f} | macro-F1 {f1:.4f}')
else:
    print('пропущено')

пропущено


In [8]:
# Экспорт для Android: ONNX int8 + vocab.txt + labels.json
import shutil
from pathlib import Path

model = model.to('cpu').eval()
EXP = '/content/export'
shutil.rmtree(EXP, ignore_errors=True); Path(EXP).mkdir(parents=True, exist_ok=True)

vocab = tok.get_vocab()
with open(f'{EXP}/vocab.txt', 'w', encoding='utf-8') as f:
    for token, _id in sorted(vocab.items(), key=lambda kv: kv[1]):
        f.write(token + '\n')
json.dump({'categories': CATS, 'max_length': MAXLEN, 'model': MODEL},
          open(f'{EXP}/labels.json', 'w'), ensure_ascii=False, indent=2)

dummy = tok(['проверка экспорта модели'], return_tensors='pt', max_length=MAXLEN,
            padding='max_length', truncation=True)
# ПОРЯДОК ВАЖЕН: BatchEncoding отдаёт ключи как (input_ids, token_type_ids, attention_mask),
# а forward у BERT ждёт (input_ids, attention_mask, token_type_ids). Если передать
# tuple(dummy.values()), маска и типы токенов меняются местами и модель слепнет.
ORDER = [k for k in ('input_ids', 'attention_mask', 'token_type_ids') if k in dummy]
kwargs = dict(input_names=ORDER, output_names=['logits'],
              dynamic_axes={k: {0: 'batch', 1: 'seq'} for k in ORDER}, opset_version=14)
try:
    torch.onnx.export(model, tuple(dummy[k] for k in ORDER), f'{EXP}/model_fp32.onnx', dynamo=False, **kwargs)
except TypeError:
    torch.onnx.export(model, tuple(dummy[k] for k in ORDER), f'{EXP}/model_fp32.onnx', **kwargs)

# Диагностика весов: если после дообучения где-то выросли выбросы, динамическое
# квантование одним масштабом на тензор «схлопывает» остальные веса, и int8
# начинает отвечать наугад. Смотрим максимум до квантования.
mx = max(float(p.abs().max()) for p in model.parameters())
print(f'максимум |вес| по модели: {mx:.3f}')
for name, par in list(model.named_parameters())[:4]:
    print(f'  {name:56s} max {float(par.abs().max()):.3f}')

import onnxruntime as ort
from onnxruntime.quantization import quantize_dynamic, QuantType

def _preds(path, texts, inames):
    sess = ort.InferenceSession(path, providers=['CPUExecutionProvider'])
    out = []
    for t in texts:
        e = tok([t], max_length=MAXLEN, padding='max_length', truncation=True, return_tensors='np')
        f = {k: v.astype(np.int64) for k, v in e.items() if k in inames}
        out.append(int(sess.run(None, f)[0][0].argmax()))
    return out

fp32 = f'{EXP}/model_fp32.onnx'
INAMES = [i.name for i in ort.InferenceSession(fp32, providers=['CPUExecutionProvider']).get_inputs()]
texts = [f"{r['title']}. {r['description']}" if r['description'] else r['title'] for r in splits['test'][:120]]
ref = _preds(fp32, texts, INAMES)

# Раньше стояло quantize_dynamic(..., weight_type=QInt8) без per_channel: один
# масштаб на весь тензор веса. После дообучения это перестало держать качество —
# int8 совпадал с PyTorch в 5 случаях из 80. Теперь делаем оба варианта с
# поканальным масштабом и берём тот, что ближе к fp32-графу.
cands = {}
for wt in (QuantType.QInt8, QuantType.QUInt8):
    path = f'{EXP}/cand_{wt.name}.onnx'
    try:
        quantize_dynamic(fp32, path, weight_type=wt, per_channel=True)
    except Exception as e:
        print(f'  {wt.name}: не удалось ({e})')
        continue
    agree = sum(a == b for a, b in zip(ref, _preds(path, texts, INAMES))) / len(texts)
    print(f'  {wt.name} per_channel: совпадение с fp32 {agree:.3f}')
    cands[wt.name] = (agree, path)

assert cands, 'ни один вариант квантования не собрался'
best_name = max(cands, key=lambda k: cands[k][0])
best_agree, best_path = cands[best_name]
shutil.move(best_path, f'{EXP}/model_int8.onnx')
for name, (_, path) in cands.items():
    if os.path.exists(path):
        os.remove(path)
size_mb = os.path.getsize(f'{EXP}/model_int8.onnx') / 1024 / 1024
print(f'model_int8.onnx: {size_mb:.0f} МБ, выбран {best_name}, совпадение с fp32 {best_agree:.3f}')
print('fp32-граф оставлен в /content/export для проверки в следующей ячейке')

/tmp/ipykernel_1177/1423485620.py:25: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(model, tuple(dummy[k] for k in ORDER), f'{EXP}/model_fp32.onnx', dynamo=False, **kwargs)
/usr/local/lib/python3.13/dist-packages/transformers/modeling_attn_mask_utils.py:196: TracerWarning: torch.tensor results are registered as constants in the trace. You can safely ignore this warning if you use this function to create tensors out of constant variables that would be the same every time you call this function. In any other case, this might cause the trace to be incorrect.
  inverted_mask = torch.tensor(1.0, dtype=dtype) - expanded_mask
/tmp/ipykernel_1177

максимум |вес| по модели: 13.382
  bert.embeddings.word_embeddings.weight                   max 0.896
  bert.embeddings.position_embeddings.weight               max 0.705
  bert.embeddings.token_type_embeddings.weight             max 0.317
  bert.embeddings.LayerNorm.weight                         max 0.974


  QInt8 per_channel: совпадение с fp32 0.033


  QUInt8 per_channel: совпадение с fp32 0.992
model_int8.onnx: 171 МБ, выбран QUInt8, совпадение с fp32 0.992
fp32-граф оставлен в /content/export для проверки в следующей ячейке


In [9]:
# Проверка перед установкой в приложение. Сравниваем три модели сразу, чтобы
# было видно, ЧТО именно сломалось: экспорт в ONNX или квантование в int8.
# Раньше сравнивались только int8 и PyTorch на 80 строках — по такому замеру
# нельзя было отличить одно от другого.
import onnxruntime as ort
model.eval()

def torch_pred(text):
    enc = tok([text], return_tensors='pt', max_length=MAXLEN, padding='max_length', truncation=True)
    with torch.no_grad():
        return int(model(**enc).logits.argmax(-1))

def onnx_pred(sess, inames, text):
    # tokenizer уже возвращает батч: enc['input_ids'] имеет форму (1, MAXLEN).
    # Оборачивать его ещё раз нельзя — получится ранг 3, и ONNX откажется.
    enc = tok([text], max_length=MAXLEN, padding='max_length', truncation=True, return_tensors='np')
    feeds = {k: v.astype(np.int64) for k, v in enc.items() if k in inames}
    return int(sess.run(None, feeds)[0][0].argmax())

sess8 = ort.InferenceSession('/content/export/model_int8.onnx', providers=['CPUExecutionProvider'])
sess32 = ort.InferenceSession('/content/export/model_fp32.onnx', providers=['CPUExecutionProvider'])
I8 = [i.name for i in sess8.get_inputs()]
I32 = [i.name for i in sess32.get_inputs()]
print('входы int8:', I8)
print('входы fp32:', I32)

sample = splits['test'][:300]
same8 = same32 = 0
for r in sample:
    text = f"{r['title']}. {r['description']}" if r['description'] else r['title']
    t = torch_pred(text)
    same32 += (t == onnx_pred(sess32, I32, text))
    same8 += (t == onnx_pred(sess8, I8, text))
n = len(sample)
print(f'fp32 ONNX совпал с PyTorch: {same32}/{n} ({same32/n:.1%})')
print(f'int8 ONNX совпал с PyTorch: {same8}/{n} ({same8/n:.1%})')
if same32 < n * 0.98:
    print('ВЫВОД: расходится уже fp32 — дело в torch.onnx.export, а не в квантовании')
elif same8 < n * 0.95:
    print('ВЫВОД: fp32 верен, расходится int8 — дело в квантовании')
else:
    print('ВЫВОД: int8 повторяет PyTorch, модель можно ставить в приложение')

# Точность самого int8 на всём тесте — именно её увидит приложение
i8_texts = [f"{r['title']}. {r['description']}" if r['description'] else r['title'] for r in splits['test']]
i8_true = [LABEL[r['category']] for r in splits['test']]
i8_preds = [onnx_pred(sess8, I8, t) for t in i8_texts]
i8_acc = sum(p == t for p, t in zip(i8_preds, i8_true)) / len(i8_true)
i8_f1, i8_per = macro_f1(i8_true, i8_preds)
print(f'INT8 НА ТЕСТЕ: acc {i8_acc:.4f} | macro-F1 {i8_f1:.4f} | строк {len(i8_true)}')

metrics = {'model': MODEL, 'max_length': MAXLEN, 'best_seed': int(best_seed),
           'train_rows': len(train_rows), 'pseudo': USE_PSEUDO,
           'test_acc': acc, 'test_macro_f1': f1,
           'per_class_f1': per, 'categories': CATS,
           'int8_agreement': same8 / n, 'fp32_agreement': same32 / n,
           'int8_test_acc': i8_acc, 'int8_test_macro_f1': i8_f1,
           'int8_per_class_f1': i8_per,
           'errors': [[a, b, k] for (a, b), k in conf.most_common(20)]}
json.dump(metrics, open('/content/export/metrics.json', 'w'), ensure_ascii=False, indent=2)
print(json.dumps({k: v for k, v in metrics.items()
                  if k not in ('per_class_f1', 'categories', 'errors', 'int8_per_class_f1')},
                 ensure_ascii=False))

входы int8: ['input_ids', 'attention_mask', 'token_type_ids']
входы fp32: ['input_ids', 'attention_mask', 'token_type_ids']
fp32 ONNX совпал с PyTorch: 300/300 (100.0%)
int8 ONNX совпал с PyTorch: 297/300 (99.0%)
ВЫВОД: int8 повторяет PyTorch, модель можно ставить в приложение
INT8 НА ТЕСТЕ: acc 0.8516 | macro-F1 0.7781 | строк 903
{"model": "DeepPavlov/rubert-base-cased", "max_length": 96, "best_seed": 42, "train_rows": 7951, "pseudo": true, "test_acc": 0.8471760797342193, "test_macro_f1": 0.7716783331420657, "int8_agreement": 0.99, "fp32_agreement": 1.0, "int8_test_acc": 0.8516057585825028, "int8_test_macro_f1": 0.7780608697749964}


In [10]:
# Архив для передачи в приложение
zip_path = '/content/newsminer_model.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    for name in ('model_int8.onnx', 'vocab.txt', 'labels.json', 'metrics.json'):
        z.write(f'/content/export/{name}', name)
print('архив:', zip_path, f'{os.path.getsize(zip_path)/1024/1024:.0f} МБ')
try:
    from google.colab import files
    files.download(zip_path)
except Exception as e:
    print('скачайте вручную из панели файлов:', e)

архив: /content/newsminer_model.zip 132 МБ


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Готово

Пришлите `newsminer_model.zip` — поставлю модель в приложение, пересоберу APK
и замерю точность и скорость на телефоне. Если тест ниже 85%, скажите — сделаем
второй ноутбук с предобучением на корпусе Lenta.ru.